# Loved One Voice — Familiarity ERP Analysis

Compares neural responses to **familiar voices** (a loved one) vs **unfamiliar voices** to detect
implicit emotional/memory processing that may survive severe brain injury.

| Condition | Stimulus | Expected response |
| --- | --- | --- |
| Familiar | Loved one's voice | Enhanced N400 / P300-like positivity |
| Unfamiliar | Unknown speaker | Baseline response |

A **positive finding** is a significant amplitude difference (familiar > unfamiliar) in the 300–600 ms
window at central-parietal electrodes, significant by permutation test (p < 0.05).

**Environment:** `stimulus_software/.venv` (MNE 1.11, scipy, pandas, matplotlib)

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

SUBJECT_ID   = 'CON013'
SESSION_DATE = '2026-04-10'

ANALYSIS_ROOT = Path.cwd().resolve()
if ANALYSIS_ROOT.name != 'analysis':
    ANALYSIS_ROOT = next(
        (parent for parent in [ANALYSIS_ROOT, *ANALYSIS_ROOT.parents]
         if parent.name == 'analysis' and (parent / 'notebooks').exists()),
        ANALYSIS_ROOT,
    )
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from lib.io import DEFAULT_EEG_CHANNELS, build_subject_paths

paths = build_subject_paths(SUBJECT_ID, SESSION_DATE, analysis_root=ANALYSIS_ROOT)
EDF_PATH = paths['edf_path']
CSV_PATH = paths['csv_path']
OUT_DIR = str(paths['out_dir'])
EEG_CHANNELS = DEFAULT_EEG_CHANNELS.copy()
BAD_CHANNELS = []  # e.g. ['Fp1', 'T3']

print(f'Subject: {SUBJECT_ID}  |  EDF: {EDF_PATH}')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy import stats

from lib.io import align_stimulus_csv, load_raw_eeg_metadata
from lib.preprocessing import load_filtered_eeg

%matplotlib inline
mne.set_log_level('WARNING')
print(f'MNE {mne.__version__}')

## 3. Load EDF + Sync Alignment

In [ ]:
raw, sfreq, available_eeg = load_raw_eeg_metadata(
    EDF_PATH,
    eeg_channels=EEG_CHANNELS,
    bad_channels=BAD_CHANNELS,
    preload=False,
    verbose=False,
)

print(f'EEG channels ({len(available_eeg)}): {available_eeg}')
print(f'sfreq: {sfreq} Hz  |  duration: {raw.n_times/sfreq:.0f}s')

In [ ]:
df, time_offset = align_stimulus_csv(CSV_PATH, sfreq=sfreq, n_times=raw.n_times)
print(f'time_offset = {time_offset:.4f} s')

## 4. Preprocessing

In [ ]:
# 1–30 Hz for ERP analysis
raw_voice = load_filtered_eeg(raw, available_eeg, l_freq=1, h_freq=30, verbose=False)
print('Preprocessing done.')

## 5. Epoch

In [ ]:
fam_df   = df[df['stim_type'] == 'familiar'].copy()
unfam_df = df[df['stim_type'] == 'unfamiliar'].copy()

print(f'Familiar trials:   {len(fam_df)}')
print(f'Unfamiliar trials: {len(unfam_df)}')

if len(fam_df) == 0 or len(unfam_df) == 0:
    raise RuntimeError(
        'No familiar/unfamiliar rows found. '
        'Check stim_type values with: df["stim_type"].value_counts()'
    )

fam_events = np.column_stack([
    fam_df['start_sample'].values,
    np.zeros(len(fam_df), dtype=int),
    np.ones(len(fam_df), dtype=int)
])
unfam_events = np.column_stack([
    unfam_df['start_sample'].values,
    np.zeros(len(unfam_df), dtype=int),
    np.full(len(unfam_df), 2, dtype=int)
])
all_events = np.vstack([fam_events, unfam_events])
all_events  = all_events[all_events[:, 0].argsort()]

# Stimulus duration varies; epoch 0–4s to capture early ERP components
epochs = mne.Epochs(
    raw_voice, events=all_events,
    event_id={'familiar': 1, 'unfamiliar': 2},
    tmin=-0.2, tmax=4.0, baseline=(-0.2, 0),
    preload=True, verbose=False
)
print(f'Epochs: {len(epochs)}  ({len(epochs["familiar"])} familiar, {len(epochs["unfamiliar"])} unfamiliar)')

## 6. Compute ERPs

In [ ]:
evoked_fam   = epochs['familiar'].average()
evoked_unfam = epochs['unfamiliar'].average()
diff_evoked  = mne.combine_evoked([evoked_fam, evoked_unfam], weights=[1, -1])

ERP_CHANNELS = [ch for ch in ['Pz', 'Cz', 'Fz', 'C3', 'C4'] if ch in evoked_fam.ch_names]
WIN_SEC = (0.3, 0.6)   # 300–600 ms window for familiarity positivity

t = evoked_fam.times
win_mask = (t >= WIN_SEC[0]) & (t <= WIN_SEC[1])

print(f'Mean amplitude ({int(WIN_SEC[0]*1000)}–{int(WIN_SEC[1]*1000)} ms):')
print(f'{"Channel":<8} {"Familiar":>12} {"Unfamiliar":>12} {"Diff":>10}')
for ch in ERP_CHANNELS:
    idx = evoked_fam.ch_names.index(ch)
    fam_amp   = evoked_fam.data[idx][win_mask].mean() * 1e6
    unfam_amp = evoked_unfam.data[idx][win_mask].mean() * 1e6
    print(f'{ch:<8} {fam_amp:>12.3f} {unfam_amp:>12.3f} {fam_amp-unfam_amp:>10.3f}')

## 7. Permutation Test

Shuffle familiar/unfamiliar labels 1000× to build a null distribution.
Statistic: mean (Familiar − Unfamiliar) amplitude in 300–600 ms window at Pz.

In [ ]:
N_PERMS   = 1000
PRIM_CH   = next((ch for ch in ['Pz', 'Cz'] if ch in epochs.ch_names), epochs.ch_names[0])
ch_idx    = epochs.ch_names.index(PRIM_CH)

epochs_data = epochs.get_data()   # (n_trials, n_ch, n_times)
labels      = epochs.events[:, 2] # 1=familiar, 2=unfamiliar
t_ep        = epochs.times
win_ep      = (t_ep >= WIN_SEC[0]) & (t_ep <= WIN_SEC[1])

def fam_amp(data, labs):
    fam_mean   = data[labs == 1, ch_idx][:, win_ep].mean()
    unfam_mean = data[labs == 2, ch_idx][:, win_ep].mean()
    return (fam_mean - unfam_mean) * 1e6

observed_diff = fam_amp(epochs_data, labels)

rng  = np.random.default_rng(42)
null = []
print(f'Running {N_PERMS} permutations on {PRIM_CH}...')
for _ in range(N_PERMS):
    null.append(fam_amp(epochs_data, rng.permutation(labels)))

null  = np.array(null)
p_val = np.mean(null >= observed_diff)

print(f'\nResults at {PRIM_CH}:')
print(f'  Observed Fam−Unfam:  {observed_diff:+.3f} µV')
print(f'  Null mean:           {null.mean():+.3f} µV  (95th pct: {np.percentile(null, 95):.3f} µV)')
print(f'  p-value:             {p_val:.4f}  {"✓ Significant" if p_val < 0.05 else "(not significant)"} (p < 0.05)')

## 8. Plots

In [ ]:
times_ms = evoked_fam.times * 1000

fig, axes = plt.subplots(len(ERP_CHANNELS), 1,
                         figsize=(10, 3 * len(ERP_CHANNELS)), sharex=True)
if len(ERP_CHANNELS) == 1:
    axes = [axes]

for ax, ch_name in zip(axes, ERP_CHANNELS):
    idx = evoked_fam.ch_names.index(ch_name)
    fam_uv   = evoked_fam.data[idx] * 1e6
    unfam_uv = evoked_unfam.data[idx] * 1e6
    diff_uv  = fam_uv - unfam_uv
    ax.plot(times_ms, unfam_uv, color='steelblue', lw=1.5, label='Unfamiliar', alpha=0.85)
    ax.plot(times_ms, fam_uv,   color='firebrick',  lw=1.5, label='Familiar',   alpha=0.85)
    ax.plot(times_ms, diff_uv,  color='darkgreen',  lw=1.5, ls='--', label='Fam − Unfam')
    ax.axvline(0, color='k', lw=0.8, ls=':')
    ax.axhline(0, color='k', lw=0.5)
    ax.axvspan(300, 600, color='gold', alpha=0.15, label='Analysis window')
    if ch_name == PRIM_CH:
        ax.set_title(f'{ch_name}  (p={p_val:.3f}{"  ✓" if p_val < 0.05 else ""})', fontsize=10)
    else:
        ax.set_title(ch_name, fontsize=10)
    ax.set_ylabel('µV')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (ms)')
fig.suptitle(f'{SUBJECT_ID} — Loved One Voice: Familiar vs Unfamiliar ERP', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_voice_erp.png'), dpi=150)
plt.show()

In [ ]:
# Null distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(null, bins=40, color='steelblue', alpha=0.7, label='Null distribution')
ax.axvline(observed_diff, color='firebrick', lw=2, label=f'Observed: {observed_diff:.2f} µV')
ax.axvline(np.percentile(null, 95), color='k', lw=1, ls='--', label='95th pct')
ax.set(xlabel='Familiar − Unfamiliar amplitude (µV)', ylabel='Count',
       title=f'{SUBJECT_ID} — Permutation null at {PRIM_CH}  (p={p_val:.3f})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_voice_null.png'), dpi=150)
plt.show()

In [ ]:
# Topomap of Familiar − Unfamiliar
montage = mne.channels.make_standard_montage('standard_1020')
diff_evoked.set_montage(montage, match_case=False, on_missing='warn')

fig = diff_evoked.plot_topomap(
    times=[0.3, 0.4, 0.5, 0.6], scalings=dict(eeg=1e6),
    show=False, cmap='RdBu_r'
)
fig.suptitle(f'{SUBJECT_ID} — Familiar − Unfamiliar topomap (µV)', fontsize=12)
fig.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_voice_topomap.png'), dpi=150)
plt.show()

## 9. Summary

**Positive finding:** Familiar > Unfamiliar amplitude (p < 0.05) in 300–600 ms window at Pz / Cz.  
**Clinical interpretation:** Preserved implicit memory / emotional processing of a familiar person's voice —
suggests residual awareness even without behavioral response.